In [12]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [13]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [14]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [15]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [16]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [17]:
square_root.invoke({"x": 467})

21.61018278497431

In [18]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [19]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-v4-pro",
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

In [20]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


我来帮您查询杭州未来几天的天气。Current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州接下来几天的天气情况如下：

- **当前气温**：22°C
- **未来5天天气**：晴天 ☀️

总体来看，杭州未来几天天气晴好，非常适合出行和户外活动。不过早晚温差可能较大，建议您根据实际体感适当添减衣物。

In [22]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少?
None
================================== Ai Message ==================================

我来计算467和529的平方根。
Tool Calls:
  square_root (call_00_V6LlNxlKHwVcIMCUrAta0462)
 Call ID: call_00_V6LlNxlKHwVcIMCUrAta0462
  Args:
    x: 467
  square_root (call_01_Ewz0YA6qJc9jGxXDp4YV5258)
 Call ID: call_01_Ewz0YA6qJc9jGxXDp4YV5258
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算结果如下：

- **467 的平方根**：约 **21.6102**（`√467 ≈ 21.61018278497431`）
- **529 的平方根**：精确为 **23**（因为 23² = 529，`√529 = 23`）
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [23]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [24]:
search_tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.douyin.com/search/%E8%92%B8%E8%9A%8C%E4%BB%80%E4%B9%88%E6%A2%97',
   'title': '蒸蚌什么梗 - 抖音',
   'content': '00:35 蒸蚌的本质是典型的空耳谐音梗，它既没有长久的生命力，同时也深埋了语言误导的隐患。而作为一名播音生更要谨慎使用。 00:43 蒸蚌跟早年间爆火的雨女无瓜',
   'score': 0.82694983,
   'raw_content': None},
  {'url': 'https://search.bilibili.com/all?keyword=%E8%92%B8%E8%9A%8C',
   'title': '蒸蚌-哔哩哔哩_Bilibili',
   'content': '### 《《《《 *蒸蚌*！！. ### 这我弟弟干的*蒸蚌*. ### 【梗百科】萝卜纸巾猫是啥梗？真棒！. ### 干的真棒，不过是循环两小时. ### *蒸蚌*四分钟纯享版. ### *蒸蚌*！！. 蒸蚌！赛博的古风也是吹到敦煌壁画了｜刘宇《壁上观》直拍. ### 荷兰猪居然真的会玩套圈？！三天就学会了！. ![[蚌曲]蒸蚌飞（虫儿飞）](//i0.hdslb.com/bfs/archive/9e3c70c4236d4b8f42480f9db24587eaaf32888f.jpg@672w_378h_1c_! ### [蚌曲]*蒸蚌*飞（虫儿飞）. ### ⚡乐 意 *蒸* *蚌*⚡. ### 巨无霸象拔蚌，迄今为止吃过最好吃的刺身，让我无法自拔. ### 阴婷生吃象拔蚌，今天吃一个小的. ### 这小子最精了，都这种情况了还想着荡秋千. ### 还是第一次看一个完整的猫打架，我能搬个凳子坐这里看一天. ### 🐷猪厂销售 但不卖猪. ### 指鹅为鸭：一只鸭腿如何骗过清北学子十几年. ### 飞起来(补档）. ### 【三国花】黢黑辣椒+龙息辣椒小合集. ### 百草敷. ### 田文镜名场面哈哈哈. 淮南“内裤姐”事件升级

In [25]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-v4-pro",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [26]:
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_00_f8vHpP8obtn4GuVrWHWo1565)
 Call ID: call_00_f8vHpP8obtn4GuVrWHWo1565
  Args:
    query: 蒸蚌 是什么梗
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://search.bilibili.com/all?keyword=%E8%92%B8%E8%9A%8C", "title": "蒸蚌-哔哩哔哩_Bilibili", "content": "《《《《  蒸蚌！！\n\n### 《《《《 蒸蚌！！\n\n直播中这我弟弟干的蒸蚌\n\n### 这我弟弟干的蒸蚌\n\n【梗百科】萝卜纸巾猫是啥梗？真棒！\n\n### 【梗百科】萝卜纸巾猫是啥梗？真棒！\n\n干的真棒，不过是循环两小时\n\n### 干的真棒，不过是循环两小时\n\n蒸蚌四分钟纯享版\n\n### 蒸蚌四分钟纯享版\n\n蒸蚌！！\n\n### 蒸蚌！！\n\n蒸蚌！赛博的古风也是吹到敦煌壁画了｜刘宇《壁上观》直拍\n\n### 蒸蚌！赛博的古风也是吹到敦煌壁画了｜刘宇《壁上观》直拍\n\n荷兰猪居然真的会玩套圈？！三天就学会了！\n\n### 荷兰猪居然真的会玩套圈？！三天就学会了！\n\n![[蚌曲]蒸蚌飞（虫儿飞）](//i0.hdslb.com/bfs/archive/9e3c70c42

## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [27]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [28]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [29]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [30]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='**"蒸蚌"** 是一个网络流行语，也是经典的**谐音梗**，它是**"真棒"（zhēn bàng）的空耳/谐音写法**。\n\n**含义：**\n- "蒸蚌"读起来和"真棒"完全一致，意思也是"真棒"、"真的很不错"。\n\n**来源与流行：**\n- 这是一个"空耳谐音梗"，类似于早期的"雨女无瓜"（与你无关）等网络用语。网友故意用看起来风马牛不相及的字来代替原词，以达到幽默、搞怪的效果。\n- "蒸蚌"的字面意思是"蒸煮蚌壳"，与"真棒"的本意形成强烈的反差萌，因此被广泛用在弹幕、评论区等场景中。\n- 在B站、抖音等平台，这个词也被用于一些萌宠视频（比如猫咪做出可爱举动时，弹幕刷"蒸蚌"）、搞笑视频的评论区等。\n\n**用法示例：**\n- 看到别人做了一件很厉害的事 → "你蒸蚌！"\n- 夸赞一个视频/作品 → "这也太蒸蚌了！"' reference=[Reference(title='蒸蚌什么梗 - 抖音', url='https://www.douyin.com/search/%E8%92%B8%E8%9A%8C%E4%BB%80%E4%B9%88%E6%A2%97'), Reference(title='我蒸蚌是什麼意思？ - HiNative', url='https://tw.hinative.com/questions/15904858'), Reference(title='蒸蚌-哔哩哔哩_Bilibili', url='https://search.bilibili.com/all?keyword=%E8%92%B8%E8%9A%8C')]
